In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')
import viewer
import dianne
import pandas as pd
import scanpy as sc
import json
dianne.setNotebookWidth(100)

In [ ]:
samples = ['JDC-WP-012-w']
vdatapath = '/projects/activities/kappsen-tmc/USERS/domans/differential-annotator-dev/visium/JDC-WP-012-w'
imgs = {samples[0]: f'{vdatapath}/cropped-bio.ome.tiff'}

adv = sc.read_10x_h5(f'{vdatapath}/filtered_feature_bc_matrix.h5')
adv.var = pd.DataFrame(index=adv.var.index)
sc.pp.log1p(adv)
df_spatial = pd.read_csv(f'{vdatapath}/spatial/tissue_positions.csv', index_col=0)
df_spatial = df_spatial.loc[adv.obs.index]
assert (adv.obs.index==df_spatial.index).all()
adv.obs = df_spatial.rename({'pxl_col_in_fullres': 'x', 'pxl_row_in_fullres': 'y'}, axis=1)[['x', 'y']]
with open(f'{vdatapath}/spatial/scalefactors_json.json', 'r') as tempfile:
    spot_size = int(json.loads(tempfile.read())['spot_diameter_fullres'])
adv.uns['spot_size'] = spot_size

visium_ads = {samples[0]: adv}

drawings = viewer.create_viewer(samples, imgs, height="800px", visium_ads=visium_ads)[1]